# GPS points — all sites
PyGMT map of GPS points from `gps_data_all_sites/`, colored by group.

Run with the **trav** conda env kernel (that's where pygmt is installed).

In [ ]:
import glob, os
import pandas as pd
import pygmt

In [ ]:
data_dir = 'gps_data_all_sites'
frames = []
for fp in sorted(glob.glob(os.path.join(data_dir, '*.csv'))):
    df = pd.read_csv(fp)
    df['group'] = os.path.splitext(os.path.basename(fp))[0]
    frames.append(df[['group', 'Latitude', 'Longitude']])
gps = pd.concat(frames, ignore_index=True).dropna(subset=['Latitude', 'Longitude'])
print(gps.groupby('group').size())
gps.head()

In [ ]:
pad = 0.01
region = [
    gps.Longitude.min() - pad,
    gps.Longitude.max() + pad,
    gps.Latitude.min() - pad,
    gps.Latitude.max() + pad,
]

grid = pygmt.datasets.load_earth_relief(resolution='03s', region=region)

fig = pygmt.Figure()
fig.grdimage(grid=grid, projection='M6i', cmap='geo', shading=True, frame='a')

colors = ['red', 'blue', 'green', 'orange', 'purple', 'cyan']
for color, (grp, sub) in zip(colors, gps.groupby('group')):
    fig.plot(
        x=sub.Longitude,
        y=sub.Latitude,
        style='c0.18c',
        fill=color,
        pen='0.5p,black',
        label=grp,
    )

fig.legend(position='JTR+jTR+o0.2c', box='+gwhite+p0.5p')
fig.show()